# Notebook de Pipeline pour Segmentation 3D Médicale

Ce notebook guide l'utilisateur à travers les étapes complètes de la pipeline :
- **Prétraitement** : Conversion des données NIfTI en tensors PyTorch
- **Entraînement** : Entraînement du modèle SegFormer3D
- **Inférence** : Prédiction sur les données de test
- **Visualisation** : Analyse des résultats avec métriques et visualisations

Assurez-vous d'avoir placé vos données brutes dans le répertoire approprié avant de commencer.

## Installation des Dépendances

Installez les packages requis depuis requirements.txt

In [ ]:
# Installation des dépendances
import subprocess
import sys

# Installer les requirements
result = subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'], 
                       capture_output=True, text=True)
print("Sortie:", result.stdout)
if result.stderr:
    print("Erreurs:", result.stderr)
print("Installation terminée!")

## Importation des Bibliothèques

Importez les bibliothèques nécessaires pour la pipeline

In [ ]:
import os
import sys
import subprocess
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display, HTML

# Configuration pour les plots
plt.style.use('default')
sns.set_palette("husl")

print("Bibliothèques importées avec succès!")

## Configuration de la Pipeline

Définissez les paramètres de configuration pour la pipeline. Modifiez ces valeurs selon vos besoins.

In [ ]:
# Configuration de la pipeline
CONFIG = {
    # Chemins
    'raw_data_dir': 'D:\\data\\documents\\ING3_2025-2026\\Projet_PIC\\prostate_raw_data_binaire',  # Répertoire des données brutes NIfTI
    'preprocessed_data_dir': './preprocessed_data/preprocessed_data_128_128_128',  # Répertoire des données prétraitées
    'config_dir': './configs/',
    'checkpoint_dir': './checkpoints/',
    'results_dir': './results/',
    'visualizations_dir': './visualizations/',
    
    # Paramètres de prétraitement
    'target_size': 128,  # Taille des volumes (64, 96, 128, 256)
    'normalize_method': 'zscore',  # 'minmax' ou 'zscore'
    'skip_existing': False,
    
    # Paramètres des splits
    'split_type': 'fixed',  # 'fixed' ou 'kfold'
    'train_ratio': 0.6,
    'val_ratio': 0.2,
    'test_ratio': 0.2,
    'k_folds': 5,
    'random_seed': 42,
    'stratified': True,
    
    # Architectures à traiter
    'architectures': ['SegFormer3D'], # Liste des architectures à entraîner
    
    # Paramètres d'entraînement
    'num_epochs': 100,
    'batch_size': 2,
    'learning_rate': 0.001,
    'device': 'cpu',  # 'cpu' ou 'cuda'
}

## Étape 1: Prétraitement des Données

Cette étape convertit les données NIfTI brutes en tensors PyTorch, effectue le resampling et la normalisation.

In [ ]:
# Fonction utilitaire pour exécuter des commandes
def run_command(command, cwd=None, description=""):
    print(f"\n=== {description} ===")
    print(f"Commande: {' '.join(command) if isinstance(command, list) else command}")
    try:
        result = subprocess.run(command, shell=True, cwd=cwd, capture_output=True, text=True, check=True)
        print("✓ Succès!")
        if result.stdout:
            print("Sortie:", result.stdout[-500:])  # Derniers 500 caractères
        return True
    except subprocess.CalledProcessError as e:
        print(f"✗ Échec: {e}")
        if e.stdout:
            print("Sortie:", e.stdout[-500:])
        if e.stderr:
            print("Erreurs:", e.stderr[-500:])
        return False

# Prétraitement pour chaque architecture
for arch in CONFIG['architectures']:
    print(f"\n{'='*50}")
    print(f"PRÉTRAITEMENT POUR {arch}")
    print(f"{'='*50}")
    
    arch_path = Path(arch)
    if not arch_path.exists():
        print(f"Architecture {arch} non trouvée, ignorée")
        continue
    
    # Script de prétraitement
    preprocess_script = arch_path / "data" / "prostate_raw_data" / "prostate_preprocess.py"
    if not preprocess_script.exists():
        print(f"Script de prétraitement non trouvé pour {arch}")
        continue
    
    # Commande de prétraitement
    output_dir = Path(CONFIG['preprocessed_data_dir']) / arch
    output_dir.mkdir(exist_ok=True)
    
    command = [
        sys.executable, str(preprocess_script),
        "--input_dir", CONFIG['raw_data_dir'],
        "--output_dir", str(output_dir),
        "--target_size", str(CONFIG['target_size']),
        "--normalize_method", CONFIG['normalize_method']
    ]
    
    if CONFIG['skip_existing']:
        command.append("--skip_existing")
    
    success = run_command(command, description=f"Prétraitement {arch}")
    if not success:
        print(f"Échec du prétraitement pour {arch}")
        continue
    
    # Générer les splits CSV
    splits_script = arch_path / "data" / "prostate_raw_data" / "create_prostate_splits.py"
    if splits_script.exists():
        if CONFIG['split_type'] == 'fixed':
            test_size = CONFIG['val_ratio']
            splits_cmd = [
                sys.executable, str(splits_script),
                "--input_dir", str(output_dir),
                "--output_dir", str(output_dir),
                "--test_size", str(test_size),
                "--random_state", str(CONFIG['random_seed']),
                "--stratified", "true" if CONFIG['stratified'] else "false"
            ]
        else:  # kfold
            splits_cmd = [
                sys.executable, str(splits_script),
                "--input_dir", str(output_dir),
                "--output_dir", str(output_dir),
                "--kfold", str(CONFIG['k_folds']),
                "--random_state", str(CONFIG['random_seed']),
                "--stratified", "true" if CONFIG['stratified'] else "false"
            ]
        
        run_command(splits_cmd, description=f"Génération des splits pour {arch}")
    
    print(f"Prétraitement terminé pour {arch}")

print("\nPrétraitement terminé pour toutes les architectures!")

## Étape 2: Entraînement des Modèles

Entraînez les modèles pour chaque architecture en utilisant les données prétraitées.

In [ ]:
# Entraînement pour chaque architecture
for arch in CONFIG['architectures']:
    print(f"\n{'='*50}")
    print(f"ENTRAÎNEMENT POUR {arch}")
    print(f"{'='*50}")
    
    arch_path = Path(arch)
    if not arch_path.exists():
        print(f"Architecture {arch} non trouvée, ignorée")
        continue
    
    # Script d'entraînement
    train_script = "train_scripts/trainer_ddp.py"
    if not (arch_path / train_script).exists():
        print(f"Script d'entraînement non trouvé pour {arch}")
        continue
    
    # Configuration
    config_path = Path(CONFIG['config_dir']) / f"config_{arch.lower()}.yaml"
    if not config_path.exists():
        print(f"Fichier de configuration non trouvé: {config_path}")
        continue
    
    # Commande d'entraînement
    python_exe = Path("..") / ".venv" / "Scripts" / "python.exe" if os.name == 'nt' else sys.executable
    command = [
        str(python_exe), train_script,
        "--config", f"../{config_path}"
    ]
    
    print(f"\n=== Entraînement {arch} ===")
    print(f"Commande: {' '.join(command)}")
    try:
        result = subprocess.run(command, shell=True, cwd=str(arch_path), check=True, capture_output=True, text=True)
        print("✓ Succès!")
        print(f"Entraînement terminé pour {arch}")
    except subprocess.CalledProcessError as e:
        print(f"✗ Échec: {e}")
        if e.stdout:
            print("Sortie:", e.stdout[-1000:])
        if e.stderr:
            print("Erreurs:", e.stderr[-1000:])
        print(f"Échec de l'entraînement pour {arch}")

print("\nEntraînement terminé pour toutes les architectures!")

## Étape 3: Inférence sur les Données de Test

Exécutez l'inférence sur les données de test en utilisant les checkpoints entraînés.

In [ ]:
# Inférence pour chaque architecture
for arch in CONFIG['architectures']:
    print(f"\n{'='*50}")
    print(f"INFÉRENCE POUR {arch}")
    print(f"{'='*50}")
    
    arch_path = Path(arch)
    if not arch_path.exists():
        print(f"Architecture {arch} non trouvée, ignorée")
        continue
    
    # Script d'inférence
    inference_script = arch_path / "inference_simple.py"
    if not inference_script.exists():
        print(f"Script d'inférence non trouvé pour {arch}")
        continue
    
    # Configuration et checkpoint
    config_path = Path(CONFIG['config_dir']) / f"config_{arch.lower()}.yaml"
    checkpoint_dir = Path(CONFIG['checkpoint_dir']) / arch
    
    if not config_path.exists():
        print(f"Configuration non trouvée: {config_path}")
        continue
    
    if not checkpoint_dir.exists():
        print(f"Répertoire de checkpoints non trouvé: {checkpoint_dir}")
        continue
    
    # Trouver le checkpoint le plus récent
    checkpoints = list(checkpoint_dir.glob("*.pth")) + list(checkpoint_dir.glob("*.pt"))
    if not checkpoints:
        print(f"Aucun checkpoint trouvé dans {checkpoint_dir}")
        continue
    
    checkpoint_path = max(checkpoints, key=lambda p: p.stat().st_mtime)
    print(f"Utilisation du checkpoint: {checkpoint_path}")
    
    # Données de test
    test_data_dir = Path(CONFIG['preprocessed_data_dir']) / arch
    if not test_data_dir.exists():
        print(f"Données de test non trouvées: {test_data_dir}")
        continue
    
    # Traiter chaque patient
    patients = [p for p in test_data_dir.iterdir() if p.is_dir()]
    if not patients:
        print(f"Aucun patient trouvé dans {test_data_dir}")
        continue
    
    results_dir = Path(CONFIG['results_dir']) / arch
    results_dir.mkdir(exist_ok=True)
    
    success_count = 0
    for patient_dir in patients:
        patient_output_dir = results_dir / patient_dir.name
        patient_output_dir.mkdir(exist_ok=True)
        
        # Chemins relatifs
        relative_config = Path("..") / config_path
        relative_checkpoint = Path("..") / checkpoint_path
        relative_input = Path("..") / patient_dir
        relative_output = Path("..") / patient_output_dir
        
        command = [
            sys.executable, str(inference_script),
            "--config", str(relative_config),
            "--checkpoint", str(relative_checkpoint),
            "--input_dir", str(relative_input),
            "--output_dir", str(relative_output)
        ]
        
        if run_command(command, cwd=str(arch_path), description=f"Inférence {arch} - {patient_dir.name}"):
            success_count += 1
    
    print(f"Inférence terminée pour {arch}: {success_count}/{len(patients)} patients traités")

print("\nInférence terminée pour toutes les architectures!")

## Étape 4: Visualisation des Résultats

Analysez les résultats avec des visualisations détaillées et des métriques.

In [ ]:
# Fonction pour charger et afficher les métriques
def load_and_display_metrics(results_dir, arch):
    """Charge et affiche les métriques d'un patient"""
    metrics_file = results_dir / "metrics.json"
    if metrics_file.exists():
        import json
        with open(metrics_file, 'r') as f:
            metrics = json.load(f)
        
        print(f"\nMétriques pour {arch}:")
        for key, value in metrics.items():
            if isinstance(value, (int, float)):
                print(f"  {key}: {value:.4f}")
            else:
                print(f"  {key}: {value}")
        return metrics
    return None

# Visualisation pour chaque architecture
for arch in CONFIG['architectures']:
    print(f"\n{'='*50}")
    print(f"VISUALISATION POUR {arch}")
    print(f"{'='*50}")
    
    arch_path = Path(arch)
    results_dir = Path(CONFIG['results_dir']) / arch
    vis_dir = Path(CONFIG['visualizations_dir']) / arch
    vis_dir.mkdir(exist_ok=True)
    
    if not results_dir.exists():
        print(f"Résultats non trouvés pour {arch}")
        continue
    
    # Script de visualisation
    vis_script = arch_path / "visualize_results.py"
    if not vis_script.exists():
        print(f"Script de visualisation non trouvé pour {arch}")
        continue
    
    # Traiter quelques patients d'exemple
    patients = [p for p in results_dir.iterdir() if p.is_dir()]
    if not patients:
        print(f"Aucun résultat de patient trouvé pour {arch}")
        continue
    
    # Prendre les 3 premiers patients pour l'exemple
    example_patients = patients[:3]
    
    for patient_dir in example_patients:
        print(f"\nGénération des visualisations pour {patient_dir.name}...")
        
        # Trouver le fichier de prédiction
        pred_files = list(patient_dir.glob("*.pt")) + list(patient_dir.glob("*.pth"))
        if not pred_files:
            print(f"Aucun fichier de prédiction trouvé pour {patient_dir.name}")
            continue
        
        pred_file = pred_files[0]
        input_dir = Path(CONFIG['preprocessed_data_dir']) / arch / patient_dir.name
        
        if not input_dir.exists():
            print(f"Données d'entrée non trouvées pour {patient_dir.name}")
            continue
        
        patient_vis_dir = vis_dir / patient_dir.name
        patient_vis_dir.mkdir(exist_ok=True)
        
        # Commande de visualisation
        command = [
            sys.executable, str(vis_script),
            "--prediction", str(pred_file),
            "--input_dir", str(input_dir),
            "--output_dir", str(patient_vis_dir),
            "--compute_errors"
        ]
        
        run_command(command, description=f"Visualisation {arch} - {patient_dir.name}")
        
        # Afficher les métriques si disponibles
        load_and_display_metrics(patient_vis_dir, f"{arch} - {patient_dir.name}")
    
    print(f"Visualisations générées pour {arch}")

print("\nVisualisation terminée pour toutes les architectures!")
print(f"\nConsultez le répertoire {CONFIG['visualizations_dir']} pour voir les résultats.")

## Analyse des Résultats

Maintenant que les visualisations sont générées, vous pouvez analyser les performances des différentes architectures.

In [ ]:
# Analyse comparative des métriques
import json
from pathlib import Path

def collect_metrics(visualizations_dir):
    """Collecte les métriques de toutes les architectures"""
    all_metrics = {}
    
    for arch_dir in Path(visualizations_dir).iterdir():
        if not arch_dir.is_dir():
            continue
        
        arch = arch_dir.name
        all_metrics[arch] = {}
        
        for patient_dir in arch_dir.iterdir():
            if not patient_dir.is_dir():
                continue
            
            metrics_file = patient_dir / "metrics.json"
            if metrics_file.exists():
                with open(metrics_file, 'r') as f:
                    patient_metrics = json.load(f)
                all_metrics[arch][patient_dir.name] = patient_metrics
    
    return all_metrics

# Collecter les métriques
metrics_data = collect_metrics(CONFIG['visualizations_dir'])

if metrics_data:
    print("Métriques collectées:")
    for arch, patients in metrics_data.items():
        print(f"\n{arch}:")
        for patient, metrics in patients.items():
            dice = metrics.get('dice', 'N/A')
            iou = metrics.get('iou', 'N/A')
            print(f"  {patient}: Dice={dice}, IoU={iou}")
else:
    print("Aucune métrique trouvée. Assurez-vous d'avoir exécuté l'étape de visualisation.")

print("\nAnalyse terminée!")

## Résumé

Vous avez terminé toutes les étapes de la pipeline :

1. ✅ **Prétraitement** : Données NIfTI converties en tensors PyTorch
2. ✅ **Entraînement** : Modèles entraînés sur les données prétraitées
3. ✅ **Inférence** : Prédictions générées sur les données de test
4. ✅ **Visualisation** : Résultats analysés avec métriques et visualisations

Les résultats sont sauvegardés dans les répertoires suivants :
- Données prétraitées : `{CONFIG['preprocessed_data_dir']}`
- Checkpoints : `{CONFIG['checkpoint_dir']}`
- Résultats d'inférence : `{CONFIG['results_dir']}`
- Visualisations : `{CONFIG['visualizations_dir']}`

Vous pouvez maintenant comparer les performances des différentes architectures et ajuster les paramètres si nécessaire.